## 1. Setup

In [ ]:
import os, subprocess
repo = '/content/Patch_Yolo'
if os.path.exists(repo):
    r = subprocess.run(['git', '-C', repo, 'pull'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Cmaddock99/Patch_Yolo.git', repo], check=True)
os.chdir(repo)
print('CWD:', os.getcwd())

In [ ]:
# Mount Google Drive — checkpoints persist here across runtime resets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics tqdm pillow opencv-python -q

## 2. Parameters — edit this cell each session

In [ ]:
# ── EDIT THESE EACH SESSION ──────────────────────────────────────────────
TARGET_MODEL   = 'yolov8n'          # yolov8n | yolo11n | yolo26n
RUN_NAME       = 'yolov8n_patch_v2' # exact output dir name (never auto-overwritten)
TARGET_EPOCH   = 1000               # raise monotonically each session; never lower
MANIFEST       = 'data/manifests/common_all_models.txt'  # 48 images, repo-relative paths
DRIVE_CKPT_DIR = '/content/drive/MyDrive/AP_checkpoints' # checkpoint survives resets here
TRAIN_MODE     = 'source_only'      # 'source_only' | 'warmstart'
WARMSTART_FROM = ''                 # patch.png path if TRAIN_MODE == 'warmstart'
LR             = 0.01               # use 0.001 for yolo26n
# ─────────────────────────────────────────────────────────────────────────

print(f'Model:   {TARGET_MODEL}')
print(f'Run:     {RUN_NAME}')
print(f'Epochs:  {TARGET_EPOCH}')
print(f'Mode:    {TRAIN_MODE}')

## 3. YOLO26 Gradient Diagnostic (run once to confirm fix is working)

In [ ]:
# Confirms that restore_v26_one2many_head() makes cv2/cv3 live
# Expected: PRE scores.grad_fn = None  →  POST scores.grad_fn = <SigmoidBackward0>
import sys; sys.path.insert(0, 'experiments')
import torch
from ultralytics import YOLO
from ultralytics_patch import restore_v26_one2many_head

m = YOLO('yolo26n.pt')
inner = m.model
detect = inner.model[-1]

# PRE-fix: cv2/cv3 are None after fuse()
print('PRE-fix  cv2:', detect.cv2, '  cv3:', detect.cv3)
dummy = torch.zeros(1, 3, 640, 640)
with torch.enable_grad():
    out = inner(dummy)
preds_dict = out if isinstance(out, dict) else (out[1] if isinstance(out, (tuple,list)) else {})
one2many = preds_dict.get('one2many', {})
scores_pre = one2many.get('scores') if isinstance(one2many, dict) else None
print('PRE-fix  scores.grad_fn:', getattr(scores_pre, 'grad_fn', 'N/A'))

# POST-fix
restore_v26_one2many_head(inner)
print('POST-fix cv2:', type(detect.cv2).__name__, '  cv3:', type(detect.cv3).__name__)
with torch.enable_grad():
    out2 = inner(dummy)
preds_dict2 = out2 if isinstance(out2, dict) else (out2[1] if isinstance(out2, (tuple,list)) else {})
one2many2 = preds_dict2.get('one2many', {})
scores_post = one2many2.get('scores') if isinstance(one2many2, dict) else None
print('POST-fix scores.grad_fn:', getattr(scores_post, 'grad_fn', 'N/A'))
del m, inner, detect
print('Diagnostic complete. ✓' if scores_post is not None and scores_post.grad_fn is not None else 'ERROR: fix not working')

## 4. Preflight — check run status before training

In [ ]:
import json, torch
from pathlib import Path

ckpt_path = Path(DRIVE_CKPT_DIR) / f'{RUN_NAME}_checkpoint.pt'
run_dir   = Path(f'outputs/{RUN_NAME}')
results_f = run_dir / 'results.json'

if results_f.exists():
    r = json.loads(results_f.read_text())
    ep = r.get('epochs', '?')
    sup = r.get('detection_suppression_pct', '?')
    print(f'COMPLETE  — {sup}% suppression, trained to epoch {ep}')
    print(f'Raise TARGET_EPOCH above {ep} and rerun the training cell to extend.')
elif ckpt_path.exists():
    c = torch.load(str(ckpt_path), map_location='cpu', weights_only=True)
    ep = c.get('epoch', '?')
    print(f'RESUMABLE — Drive checkpoint at epoch {ep} / {TARGET_EPOCH}')
elif (run_dir / 'checkpoint.pt').exists():
    c = torch.load(str(run_dir / 'checkpoint.pt'), map_location='cpu', weights_only=True)
    ep = c.get('epoch', '?')
    print(f'RESUMABLE — local checkpoint at epoch {ep} / {TARGET_EPOCH} (no Drive copy)')
else:
    print('NEW run — starting from scratch')

## 5. Training

Set parameters in §2, check status in §4, then run this cell.

> **Session protocol:** raise `TARGET_EPOCH` monotonically each session (e.g. 1000 → 1500 → 2000). Never lower it. If the run is COMPLETE, raise the target first — the script will resume and continue.

> **Warm-start:** set `TRAIN_MODE = 'warmstart'` and `WARMSTART_FROM = 'outputs/yolov8n_patch_v2/patches/patch.png'` to initialize a new model's training from an existing patch (full gradient budget, info shared via initialization).

In [ ]:
warmstart_flag = f'--load-patch {WARMSTART_FROM}' if TRAIN_MODE == 'warmstart' and WARMSTART_FROM else ''
run_name_flag  = f'--run-name {RUN_NAME}'

!python experiments/ultralytics_patch.py \\
    --model {TARGET_MODEL} \\
    {run_name_flag} \\
    --manifest {MANIFEST} \\
    --epochs {TARGET_EPOCH} \\
    --lr {LR} \\
    --tv-weight 0.05 --nps-weight 0.01 \\
    --block-erase-prob 0.5 --cutout-prob 0.3 --rot-max 15.0 \\
    --patch-size 100 --batch-size 8 --seed 42 --display 5 \\
    --checkpoint-dir {DRIVE_CKPT_DIR} \\
    --resume {warmstart_flag}

## 6. Save Artifacts to Repo

In [ ]:
# Copy final artifacts from Drive checkpoint back to outputs/ for commit
from pathlib import Path
import shutil

run_dir = Path(f'outputs/{RUN_NAME}')
print(f'Artifacts in {run_dir}:')
for f in ['patches/patch.png', 'results.json', 'loss_history.json']:
    p = run_dir / f
    print(f'  {f}: {"exists" if p.exists() else "MISSING"}')

## 7. Push to GitHub

In [ ]:
from google.colab import userdata
import subprocess
TOKEN = userdata.get('GithubPAT')  # set in Colab Secrets (key icon in left sidebar)
REPO  = 'https://Cmaddock99:' + TOKEN + '@github.com/Cmaddock99/Patch_Yolo.git'

!git config user.email 'cmaddock99@github'
!git config user.name  'Cmaddock99'

# Stage only final artifacts — never commit checkpoints or raw images
run_dir = f'outputs/{RUN_NAME}'
!git add {run_dir}/results.json {run_dir}/patches/patch.png {run_dir}/loss_history.json
!git add outputs/*/results.json  # catch any transfer eval results too
!git diff --cached --stat
!git commit -m 'results: {RUN_NAME} epoch {TARGET_EPOCH}'
!git push {REPO} main

---
## § POST-FREEZE EVALUATION

> **Run these cells only after all three source-only runs (v8n, v11n, v26n) are complete and committed.**

Transfer evals are fast (eval-only, no training). Run the full 6×6 matrix in one session after source patches are frozen.

### Transfer Matrix — all 6 directions

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo11n --eval-only \\
    --load-patch outputs/yolov8n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo26n --eval-only \\
    --load-patch outputs/yolov8n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --eval-only \\
    --load-patch outputs/yolo11n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo26n --eval-only \\
    --load-patch outputs/yolo11n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --eval-only \\
    --load-patch outputs/yolo26n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo11n --eval-only \\
    --load-patch outputs/yolo26n_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

### Optional: Joint Training (post-freeze only)

Joint training optimizes one patch against two models simultaneously (gradient budget is shared). Run only after source-only runs are frozen and transfer results are recorded.

To avoid overwriting source-only results, set `--run-name` explicitly.

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --co-model yolo11n \\
    --run-name yolov8n+yolo11n_joint_patch_v2 \\
    --manifest data/manifests/common_all_models.txt \\
    --seed 42 --epochs 1000 --lr 0.01 --tv-weight 0.05 \\
    --nps-weight 0.01 --block-erase-prob 0.5 --cutout-prob 0.3 \\
    --rot-max 15.0 --patch-size 100 --batch-size 8 --display 5 \\
    --co-weight 0.5 \\
    --checkpoint-dir {DRIVE_CKPT_DIR} --resume

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --co-model yolo26n \\
    --run-name yolov8n+yolo26n_joint_patch_v2 \\
    --manifest data/manifests/common_all_models.txt \\
    --seed 42 --epochs 1000 --lr 0.01 --tv-weight 0.05 \\
    --nps-weight 0.01 --block-erase-prob 0.5 --cutout-prob 0.3 \\
    --rot-max 15.0 --patch-size 100 --batch-size 8 --display 5 \\
    --co-weight 0.5 \\
    --checkpoint-dir {DRIVE_CKPT_DIR} --resume

#### Joint patch transfers

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --eval-only \\
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo11n --eval-only \\
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo26n --eval-only \\
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolov8n --eval-only \\
    --load-patch outputs/yolov8n+yolo26n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo11n --eval-only \\
    --load-patch outputs/yolov8n+yolo26n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

In [ ]:
!python experiments/ultralytics_patch.py \\
    --model yolo26n --eval-only \\
    --load-patch outputs/yolov8n+yolo26n_joint_patch_v2/patches/patch.png \\
    --manifest data/manifests/common_all_models.txt \\
    --conf-threshold 0.5 --display 0

### Results Summary

In [ ]:
import json
from pathlib import Path

print(f'{'Model / Patch':<40} {'Supp%':>6} {'Clean':>6} {'Patched':>8} {'ΔConf':>7}')
print('-' * 72)
for d in sorted(Path('outputs').iterdir()):
    rf = d / 'results.json'
    if not rf.exists(): continue
    r = json.loads(rf.read_text())
    supp = r.get('detection_suppression_pct', '?')
    clean = r.get('clean_person_detections', '?')
    patched = r.get('patched_person_detections', '?')
    conf_clean = r.get('mean_conf_clean', 0)
    conf_patched = r.get('mean_conf_patched', 0)
    delta = round(conf_clean - conf_patched, 4) if (conf_clean and conf_patched) else '?'
    ep = r.get('epochs', r.get('resumed_from_epoch', ''))
    name = d.name
    print(f'{name:<40} {supp:>6} {str(clean):>6} {str(patched):>8} {str(delta):>7}')

### Push Post-Freeze Results

In [ ]:
from google.colab import userdata
TOKEN = userdata.get('GithubPAT')
REPO  = 'https://Cmaddock99:' + TOKEN + '@github.com/Cmaddock99/Patch_Yolo.git'
!git config user.email 'cmaddock99@github'
!git config user.name  'Cmaddock99'
!git add outputs/*/results.json outputs/*/patches/patch.png outputs/*/loss_history.json
!git diff --cached --stat
!git commit -m 'results: post-freeze transfer matrix and optional joint patches'
!git push {REPO} main